### RAG Pipelines- Data Ingestion to Vector DB Pipeline

In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\sumit\AppData\Local\Temp\ipykernel_8764\3933654057.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
c:\Users\sumit\Desktop\30 Days\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
### Read all the pdf inside the directory

def process_all_pdfs(pdf_directory):
    """Process all pdf files in a directory"""
    
    all_documents=[]
    pdf_dir= Path(pdf_directory)
    
    #Find all pdf files recursively
    pdf_files= list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader= PyPDFLoader(str(pdf_file))
            documents= loader.load()
            
            for doc in documents:
                doc.metadata['source_file']=pdf_file.name;
                doc.metadata['file_type']='pdf'
                
            all_documents.extend(documents)
            print(f" Loaded {len(documents)} pages")
        
        except Exception as e:
            print(f" Error: {e}")
        
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

all_pdf_documents = process_all_pdfs("../data")

Found 2 PDF files to process

Processing: 11.pdf


Multiple definitions in dictionary at byte 0x2494e for key /Group
Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 12 0 (offset 0)
Ignoring wrong pointing object 15 0 (offset 0)
Ignoring wrong pointing object 24 0 (offset 0)
Ignoring wrong pointing object 26 0 (offset 0)


 Loaded 25 pages

Processing: meidtations.pdf
 Loaded 128 pages

Total documents loaded: 153


In [3]:
all_pdf_documents


[Document(metadata={'producer': 'pdfTeX-1.40.29', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-08-19T19:06:11-07:00', 'author': '', 'title': '', 'subject': '', 'keywords': '', 'moddate': '2026-08-19T19:06:11-07:00', 'trapped': '/False', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.29 (TeX Live 2026) kpathsea version 6.4.2', 'source': '..\\data\\pdf\\11.pdf', 'total_pages': 25, 'page': 0, 'page_label': '1', 'source_file': '11.pdf', 'file_type': 'pdf'}, page_content='Speech and Language Processing. Daniel Jurafsky & James H. Martin. Copyright©2026. All\nrights reserved. Draft of August 19, 2026.\nCHAPTER\n11\nInformation Retrieval and\nRetrieval-Augmented Generation\nOn two occasions I have been asked,—“Pray, Mr. Babbage, if you put into\nthe machine wrong figures, will the right answers come out?” ... I am not able\nrightly to apprehend the kind of confusion of ideas that could provoke such a\nquestion. Babbage (1864)\nPeople need to know things. So pretty

In [4]:
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    text_splitter= RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs= text_splitter.split_documents(documents)
    print(f"Split{len(documents)} documents into {len(split_docs)} chunks")
    
    #Show example of a Chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content:{split_docs[0].page_content[:200]}...")
        print(f"Metadata:{split_docs[0].metadata}")
    return split_docs

In [5]:
chunks = split_documents(all_pdf_documents)
chunks

Split153 documents into 673 chunks

Example chunk:
Content:Speech and Language Processing. Daniel Jurafsky & James H. Martin. Copyright©2026. All
rights reserved. Draft of August 19, 2026.
CHAPTER
11
Information Retrieval and
Retrieval-Augmented Generation
On...
Metadata:{'producer': 'pdfTeX-1.40.29', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-08-19T19:06:11-07:00', 'author': '', 'title': '', 'subject': '', 'keywords': '', 'moddate': '2026-08-19T19:06:11-07:00', 'trapped': '/False', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.29 (TeX Live 2026) kpathsea version 6.4.2', 'source': '..\\data\\pdf\\11.pdf', 'total_pages': 25, 'page': 0, 'page_label': '1', 'source_file': '11.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'pdfTeX-1.40.29', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-08-19T19:06:11-07:00', 'author': '', 'title': '', 'subject': '', 'keywords': '', 'moddate': '2026-08-19T19:06:11-07:00', 'trapped': '/False', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.29 (TeX Live 2026) kpathsea version 6.4.2', 'source': '..\\data\\pdf\\11.pdf', 'total_pages': 25, 'page': 0, 'page_label': '1', 'source_file': '11.pdf', 'file_type': 'pdf'}, page_content='Speech and Language Processing. Daniel Jurafsky & James H. Martin. Copyright©2026. All\nrights reserved. Draft of August 19, 2026.\nCHAPTER\n11\nInformation Retrieval and\nRetrieval-Augmented Generation\nOn two occasions I have been asked,—“Pray, Mr. Babbage, if you put into\nthe machine wrong figures, will the right answers come out?” ... I am not able\nrightly to apprehend the kind of confusion of ideas that could provoke such a\nquestion. Babbage (1864)\nPeople need to know things. So pretty

### Embedding and VectorStoreDB

In [6]:
import os

os.environ["HF_HUB_DISABLE_XET"] = "1"

import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Tuple, Dict, Any
from sklearn.metrics.pairwise import cosine_similarity


In [7]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str="all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager

        Args:
            model_name : HuggingFace model name for sentence embeddings
        """
        
        self.model_name= model_name
        self.model= None
        self._load_model()
    
    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model:{self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded Successfully. Embedding dimension:{self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model{self.model_name}:{e}")
            raise
        
    def generate_embeddings(self, texts: List[str])-> np.ndarray:
        """Generate embedding for list of Texts

        Args:
            texts: List of texts to be embedded 

        Returns:
            numpy arrays of embeddings with shape(len(texts),embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        print(f"Generating embedding for {len(texts)} texts...")
        embeddings= self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings
        
## Initialize embedding manager

embedding_manager= EmbeddingManager()
embedding_manager

Loading embedding model:all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2909.14it/s]


Model loaded Successfully. Embedding dimension:384


C:\Users\sumit\AppData\Local\Temp\ipykernel_8764\876014844.py:21: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded Successfully. Embedding dimension:{self.model.get_sentence_embedding_dimension()}")


### VectorStore

In [8]:
class VectorStore:
    """Manages document embeddings in ChromaDB vector store"""
    
    def __init__(self, collection_name: str= "pdf_documents", persist_directory: str="../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        
        self.collection_name= collection_name
        self.persist_directory= persist_directory
        self.client= None
        self.collection= None
        self._initialize_store()
    
    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise
    
    
    
    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store.

        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")

        print(f"Adding {len(documents)} documents to vector store...")

        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{i}_{uuid.uuid4().hex[:8]}"
            ids.append(doc_id)
 
            documents_text.append(doc.page_content)

            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadatas.append(metadata)

            embeddings_list.append(embedding.tolist())

        try:
            self.collection.add(
                ids=ids,
                documents=documents_text,
                metadatas=metadatas,
                embeddings=embeddings_list
            )

            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
        
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise
        
vectorstore = VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 673


In [9]:
chunks

[Document(metadata={'producer': 'pdfTeX-1.40.29', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-08-19T19:06:11-07:00', 'author': '', 'title': '', 'subject': '', 'keywords': '', 'moddate': '2026-08-19T19:06:11-07:00', 'trapped': '/False', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.29 (TeX Live 2026) kpathsea version 6.4.2', 'source': '..\\data\\pdf\\11.pdf', 'total_pages': 25, 'page': 0, 'page_label': '1', 'source_file': '11.pdf', 'file_type': 'pdf'}, page_content='Speech and Language Processing. Daniel Jurafsky & James H. Martin. Copyright©2026. All\nrights reserved. Draft of August 19, 2026.\nCHAPTER\n11\nInformation Retrieval and\nRetrieval-Augmented Generation\nOn two occasions I have been asked,—“Pray, Mr. Babbage, if you put into\nthe machine wrong figures, will the right answers come out?” ... I am not able\nrightly to apprehend the kind of confusion of ideas that could provoke such a\nquestion. Babbage (1864)\nPeople need to know things. So pretty

In [10]:
### Convert the text to embeddinngs
texts=[doc.page_content for doc in chunks]

### Generate the embeddings

embeddings= embedding_manager.generate_embeddings(texts)

## Store in the vector database

vectorstore.add_documents(chunks, embeddings)

Generating embedding for 673 texts...


Batches: 100%|██████████| 22/22 [01:09<00:00,  3.15s/it]


Generated embeddings with shape: (673, 384)
Adding 673 documents to vector store...
Successfully added 673 documents to vector store
Total documents in collection: 1346


### Retriever Pipeline from VectorStore


In [17]:
class RAGRetriever:
    """Handles query based retrieval from vector store"""
    
    def __init__(self, vectorstore: VectorStore, embedding_manager: EmbeddingManager):
        """Initialize the retriever

        Args:
            vectorstore: vector Store containig document embeddings
            embedding_manager : Manager for generatinng query embeddings
        """
        
        self.vectorstore= vectorstore
        self.embedding_manager= embedding_manager
        
    def retrieve(self, query: str, top_k: int=5, score_threshold: float=0.0)-> List[Dict[str,Any]]:
        """Retrieve relevant document from a query

        Args:
            query: the search query
            top_k: number of top result to return
            score_threshold: Minimum similarty  score threshold 

        Returns:
            List of dictionaries containg retrieved documents and metadata 
        """
        
        print(f"Retrieving documents for query:'{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        query_embedding= self.embedding_manager.generate_embeddings([query])[0]
        
        try:
            results= self.vectorstore.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs=[]
            
            if results['documents'] and results['documents'][0]:
                documents= results['documents'][0]
                metadatas=results['metadatas'][0]
                distances= results['distances'][0]
                ids = results['ids'][0]
                
                for i,(doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Chroma's default L2 distance is nonnegative; map it to a score in (0, 1].
                    similarity_score= 1 / (1 + distance)
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i+1
                        })
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
                
            return retrieved_docs
        
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []
        
rag_retriver = RAGRetriever(vectorstore, embedding_manager)

In [12]:
rag_retriver


In [19]:
rag_retriver.retrieve("may	be	reproducede")

Retrieving documents for query:'may	be	reproducede'
Top K: 5, Score threshold: 0.0
Generating embedding for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  6.33it/s]

Generated embeddings with shape: (1, 384)


Retrieved 5 documents (after filtering)


[{'id': 'doc_195_fcb985ce',
  'content': 'beauty. The hanging down of grapes, the brow of a lion, the froth of a foaming wild \nboar, and many other like things, though by themselves considered, they are far from \nany beauty, yet because they happen naturally, they both are comely, and delightful; \nso that if a man shall with a profound mind and apprehension, consider all things in \nthe world, even among all those things which are but mere accessories and natural \nappendices as it were, there will scarce appear anything unto him, wherein he will \nnot find matter of pleasure and delight. So will he behold with as much pleasure the \ntrue rictus of wild beasts, as those which by skilful painters and other artificers are \nimitated. So will he be able to perceive the proper ripeness and beauty of old age, \nwhether in man or woman: and whatsoever else it is that is beautiful and alluring in \nwhatsoever is, with chaste and continent eyes he will soon find out and discern.',
  'metada

### Integrate VectorDB Context pipeline with LLM Output

In [33]:
### Simple RAG pipeline with Groq LLM

import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")
if not groq_api_key:
    raise ValueError("Set GROQ_API_KEY in your environment or .env file before initializing ChatGroq.")

llm = ChatGroq(
    groq_api_key=groq_api_key,
    model_name="openai/gpt-oss-20b",
    temperature=0.1,
    max_tokens=1024,
)

def rag_simple(query, retriever, llm, top_k=3):
    # Retrieve relevant documents
    results = retriever.retrieve(query, top_k=top_k)
    context = "\n\n".join(doc["content"] for doc in results) if results else ""

    if not context:
        return "No relevant context found to answer the question."

    prompt = f"""Use the following context to answer the question concisely.
Context:
{context}

Question: {query}

Answer:"""
    response = llm.invoke(prompt)
    return response.content

In [34]:
answer= rag_simple("What	Is	Programming?", rag_retriver, llm)
print(answer)


Retrieving documents for query:'What	Is	Programming?'
Top K: 3, Score threshold: 0.0
Generating embedding for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 44.48it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


**Programming** is the process of designing, writing, testing, debugging, and maintaining code that instructs a computer to perform specific tasks or solve problems.


### Enhanced RAG Pipeline Features

In [43]:
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """Retrieve context, generate an answer, and return source/confidence details."""
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {
            "answer": "No relevant context found.",
            "sources": [],
            "confidence_score": 0.0,
            "context": "",
        }

    context = "\n\n".join(doc["content"] for doc in results)
    sources = [
        {
            "source": doc["metadata"].get("source_file", doc["metadata"].get("source", "unknown")),
            "page": doc["metadata"].get("page", "unknown"),
            "score": doc["similarity_score"],
            "preview": doc["content"][:300] + "...",
        }
        for doc in results
    ]
    confidence_score = max(doc["similarity_score"] for doc in results)

    prompt = f"""Use the following context to answer the question concisely.
Context:
{context}

Question: {query}

Answer:"""
    response = llm.invoke(prompt)

    output = {
        "answer": response.content,
        "sources": sources,
        "confidence_score": confidence_score,
    }
    if return_context:
        output["context"] = context
    return output


# Example usage:
result = rag_advanced("What is Programming", rag_retriver, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result["answer"])
print("Sources:", result["sources"])
print("Confidence:", result["confidence_score"])
print("Context Preview:", result["context"][:300])

Retrieving documents for query:'What is Programming'
Top K: 3, Score threshold: 0.1
Generating embedding for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.09it/s]


Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Answer: Programming is the process of designing, writing, testing, debugging, and maintaining code that instructs a computer to perform specific tasks.
Sources: [{'source': '11.pdf', 'page': 18, 'score': 0.38311769986009087, 'preview': 'solver uses each question’s attached document, or closed book if the documents are\nnot used, while datasets like MMLU are solely closed book.\nAnother dimension of variation is the format of the answer: multiple-choice\nversus freeform. And of course there are variations in prompting, like whether the...'}, {'source': '11.pdf', 'page': 18, 'score': 0.38311769986009087, 'preview': 'solver uses each question’s attached document, or closed book if the documents are\nnot used, while datasets like MMLU are solely closed book.\nAnother dimension of variation is the format of the answer: multiple-choice\nversus freeform. And of course there are variations in prompting, like whet

In [46]:
# --- Advanced RAG pipeline: Streaming, Citations, History, Summarization---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriver, llm):
        self.retriver = retriver
        self.llm = llm
        self.history = []

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2,
              stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        results = self.retriver.retrieve(question, top_k=top_k, score_threshold=min_score)

        if not results:
            response = {
                "question": question,
                "answer": "No relevant context found.",
                "sources": [],
                "summary": None,
                "history": self.history,
            }
            self.history.append({
                "question": question,
                "answer": response["answer"],
                "sources": [],
                "summary": None,
            })
            return response

        context = "\n\n".join(doc["content"] for doc in results)
        sources = [
            {
                "source": doc["metadata"].get("source_file", doc["metadata"].get("source", "unknown")),
                "page": doc["metadata"].get("page", "unknown"),
                "score": doc["similarity_score"],
                "preview": doc["content"][:300] + "...",
            }
            for doc in results
        ]

        prompt = f"""Use the following context to answer the question concisely. \nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
        if stream:
            print("Streaming answer:")
            for i in range(0, len(prompt), 80):
                print(prompt[i:i + 80], end="", flush=True)
                time.sleep(0.05)
            print()

        response = self.llm.invoke(prompt)
        answer = response.content

        citations = [f"[{i + 1}] {src['source']}(page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences: \n{answer}"
            summary_response = self.llm.invoke(summary_prompt)
            summary = summary_response.content

        self.history.append({
            "question": question,
            "answer": answer,
            "sources": sources,
            "summary": summary,
        })

        return {
            "question": question,
            "answer": answer_with_citations,
            "sources": sources,
            "summary": summary,
            "history": self.history,
        }


# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriver, llm)
result = adv_rag.query("Experimenting with BASH", top_k=3, min_score=0.3, stream=True, summarize=True)
print("\nFinal Answer:", result["answer"])
print("Summary:", result["summary"])
print("History:", result["history"][-1])

Retrieving documents for query:'Experimenting with BASH'
Top K: 3, Score threshold: 0.3
Generating embedding for 1 texts...


Batches: 100%|██████████| 1/1 [00:02<00:00,  2.03s/it]


Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely. 
Context:
ten on linen, which belongs to things sacred. Then on the gate as we came out 
was written twice, as follows: ‘Priest don the fell.’84
 I asked one of the inhabit-
ants what that word was. He said it was the word in the Hernican dialect for 
the victim’s skin, which the priest puts over his conical cap when he enters the 
city. I found out many other things which I desired to know, but the only thing 
I do not desire is that you should he absent from me; that is my chief anxiety. 
Now for yourself, when you left that place, did you go to Aurelia or to Campa-
nia? Be sure to write to me, and say whether you have opened the vintage, or 
carried a host of books to the country-house; this also, whether you miss me; I 
am foolish to ask it, whenas you tell it me of yourself. Now if you miss me and if 
you love me, send me y